In [13]:
import pandas as pd
#import package to train_test_split
from sklearn.model_selection import train_test_split
# Load dataset "./working/dataset/data_10k.csv"
data = pd.read_csv("../data/data_10k.csv")
data = data[['rating', 'title', 'text']]
# create a new column 'review' by combining 'title' and 'text'
data['review'] = data['title'].fillna('') + '\n' + data['text'].fillna('')
data = data[['rating', 'review']]

data_train, data_test = train_test_split(data, train_size = 0.8, random_state = 42, shuffle = True)
print(data.head())

   rating                                             review
0       1  One Star\ndon't fit like the picture, can't ev...
1       1  Not for thick curly hair\nUnfortunately the pr...
2       1  Washed out after one shampoo and not vibrant\n...
3       1  It’s not as good as advertised. I won’t purcha...
4       1  One Star\nDid not work when plugged it in. Got...


In [15]:
data_train.to_csv("../data/train.csv", index = False)
data_test.to_csv("../data/val.csv", index = False)

In [ ]:
# Requires transformers>=4.51.0

import torch
import torch.nn.functional as F

from torch import Tensor
from transformers import AutoTokenizer, AutoModel


def last_token_pool(last_hidden_states: Tensor,
                    attention_mask: Tensor) -> Tensor:
    """
    Pool a sequence of token embeddings into a single embedding per example
    by selecting the embedding of the last non-padded token.

    Arguments:
    - last_hidden_states: Tensor of shape (batch_size, seq_len, hidden_dim)
      returned by the transformer model.
    - attention_mask: Tensor of shape (batch_size, seq_len) with 1 for real
      tokens and 0 for padding.

    Behavior:
    - If tokenizer used left padding, the last position (index -1) is a real token
      for every sequence. Detect this case by checking if attention_mask[:, -1]
      is 1 for the whole batch. In that case simply take last_hidden_states[:, -1].
    - Otherwise (right padding), compute the index of the last real token for
      each sequence as attention_mask.sum(dim=1) - 1 and index into last_hidden_states.
    """
    # Detect whether tokenizer padded on the left by checking the last column of the mask.
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])

    if left_padding:
        # For left-padded inputs the final token position is the actual last token.
        return last_hidden_states[:, -1]
    else:
        # For right-padded inputs compute each sequence's last non-pad token index
        # (attention_mask has 1s for tokens, so sum gives length).
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        # Index the last token embedding for each batch element.
        return last_hidden_states[
            torch.arange(batch_size, device=last_hidden_states.device),
            sequence_lengths
        ]


def get_detailed_instruct(task_description: str, query: str) -> str:
    """
    Build a single string combining a one-sentence instruction and the query.
    This is useful when the model expects each query to be prefixed with an instruction.
    """
    return f'Instruct: {task_description}\nQuery:{query}'


# A one-sentence instruction describing the retrieval task.
task = 'Given a web search query, retrieve relevant passages that answer the query'

# Two example queries, each prefixed with the one-sentence instruction using helper above.
queries = [
    get_detailed_instruct(task, 'What is the capital of China?'),
    get_detailed_instruct(task, 'Explain gravity')
]

# Candidate documents (no instruction prefix required for documents here).
documents = [
    "The capital of China is Beijing.",
    "Gravity is a force that attracts two bodies towards each other. It gives weight to physical objects and is responsible for the movement of planets around the sun."
]

# Combine queries and documents into one list so they can be tokenized/batched together.
input_texts = queries + documents

# Load tokenizer and model for embeddings. This example uses a Qwen embedding model.
# Setting padding_side='left' ensures padding is added to the left of sequences.
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-Embedding-0.6B', padding_side='left')
model = AutoModel.from_pretrained('Qwen/Qwen3-Embedding-0.6B')

# Optionally one could enable flash_attention_2 and use float16 on CUDA for speed/memory:
# model = AutoModel.from_pretrained('Qwen/Qwen3-Embedding-0.6B', attn_implementation="flash_attention_2", torch_dtype=torch.float16).cuda()

max_length = 8192  # Maximum token length for tokenizer truncation/padding

# Tokenize all inputs into a single batch tensor. return_tensors="pt" gives PyTorch tensors.
batch_dict = tokenizer(
    input_texts,
    padding=True,
    truncation=True,
    max_length=max_length,
    return_tensors="pt",
)

# Move the batch tensors to the same device as the model (cpu or cuda).
batch_dict.to(model.device)

# Run the model to obtain hidden states.
outputs = model(**batch_dict)

# Pool the token-level hidden states to a single vector per input using the helper.
embeddings = last_token_pool(outputs.last_hidden_state, batch_dict['attention_mask'])

# L2-normalize embeddings so cosine similarities are simply dot-products.
embeddings = F.normalize(embeddings, p=2, dim=1)

# Compute similarity scores: dot product between the first two embeddings (queries)
# and the last two embeddings (documents). Result shape: (2 queries x 2 documents).
scores = (embeddings[:2] @ embeddings[2:].T)

# Print a Python list of similarity scores for inspection.
print(scores.tolist())
# Example output:
# [[0.7645568251609802, 0.14142508804798126], [0.13549736142158508, 0.5999549627304077]]
